[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/stokes_swimmer/stokes_swimmer.ipynb)

# Swimming without Inertia

This example computes how a swimmer moves without inertia, as a chain of beads in a viscous fluid. In slow, thick flow, a bacterium in water or a person in honey, nothing coasts: the moment a swimmer stops moving its body, it stops. At every instant the pushes its body gives the fluid balance, with no net force and no net turning. Purcell's scallop theorem follows: a swimmer that moves its body back and forth along one path ends every stroke where it began. To get anywhere it must take its shape around a loop, as Purcell's three-link swimmer does with its two hinges.

The swimmer is modelled in the projective geometric algebra of space as a field of beads, points, and the body's motion as a twist, a bivector. Over `Twist`, the space of all rates, `sites.commutator(Twist)` is the map from the body's rate to every bead's velocity, and its adjugate, `rigid.adjugate()`, sums the beads' forces into the forque on the body, with no lever arms written out. The regularized Stokeslets between every pair of beads make one map from all the beads' forces to all their velocities, and two solves give the body's rate that keeps the forque zero as the swimmer changes its shape.

In the notation of regularized Stokeslets the flow at a separation $\mathbf r$ from a force $\mathbf F$ spread over a blob of size $\epsilon$ reads as $\mathbf u = \frac{1}{8\pi\mu}\,\frac{(r^2 + 2\epsilon^2)\,\mathbf F + (\mathbf F\cdot\mathbf r)\,\mathbf r}{(r^2 + \epsilon^2)^{3/2}}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import PGA2D as ga
from examples.util.animation import save_figure, show_animation
from examples.fluids.stokes_swimmer import render

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                             # Antivector-
Twist = ga.gatype.bivector()                               # Bivector+
Forque = ga.gatype.antibivector()                          # Antibivector-
Motor = ga.gatype.rotor()                                  # Rotor+
Force = ga.gatype.vector().derive.nondegenerate            # Vector+
Direction = Force.dual().output_gatype                     # Antivector-
Mobility = ga.gatype((Direction, Force))                   # Direction- <- Force+
origin = mv.w.dual()                                       # [] Point-

## 1. The mobility map

A push on the fluid at one bead sets all of the fluid moving, less and less with distance: the flow of a force spread over a small blob, a regularized Stokeslet. Over every pair of beads, these flows make one map from the forces the beads push with to the velocities they move at. The body's own motion, a turn about some centre, gives every bead a velocity of its own, and the forces the beads push with add up to a single forque on the body. Both come from one map: the map from the body's rate to the beads' velocities, `rigid`, read backwards across the regressive product as its adjugate, sums the forces into the forque, because a forque paired with a rate is the power the forces deliver to beads moving at that rate. As the swimmer changes its shape, its body must move so that this forque stays zero, and that is one solve for the body's rate of motion.

In the matrix notation of microhydrodynamics, the map from forces to velocities is the beads' Oseen matrix, and the body's rate comes from a three by three resistance matrix.

In [ ]:
# The swimmer: the half-length of its centre link, the beads' offsets along the centre link and along each arm,
# and the radius of the blob each bead's force is spread over.
hinge_half_span = 0.8
centre_offsets = np.array([-0.5, 0.0, 0.5])
arm_offsets = np.array([0.35, 0.7, 1.05])
bead_radius = 0.14

In [ ]:
def stokeslet(targets: Point, sources: Point, radius: float) -> Mobility:
    """Fluid velocity (`Direction`) at the targets per unit blob force (`Force`) at the sources, for
    regularized Stokeslets of blob `radius`."""
    # The difference of two points of one weight is a direction; as one, the map from forces to velocities is
    # square, as its solves need.
    separations = (targets - sources).cast(Direction)                       # [...] Direction-
    # The squared distance, the squared norm of the line joining the two points, softened by the radius: the
    # blob spreads each force over the bead's size, which keeps the flow finite where the force acts.
    softened = (targets & sources).norm_squared() + radius**2               # [...] Scalar+
    # The fluid moves along the force, and further along the separation by the force's pairing with it;
    # both parts fall off with distance.
    return ((softened + radius**2) * Force.dual() + separations * (Force & separations)) / (softened * softened.square_root())


def mobility(sites: Point, radius: float) -> Mobility:
    """The flow each bead's force drives at every other bead: one map from all the beads' forces to all
    their velocities. Its blocks depend on every pair of beads, so they are formed over the beads as a
    batch and assembled into the map between the two fields."""
    beads = sites.batch()                                                   # [..., beads] Point-
    return stokeslet(beads[..., :, None], beads[..., None, :], radius).field(0, 1)   # [...] Direction-[beads] <- Force+[beads]


def swim(sites: Point, shape_velocity: Direction, radius: float) -> tuple[Twist, Force]:
    """The body-frame rigid rate (`Twist`) and bead forces (`Force[beads]`) of a force- and torque-free swimmer."""
    bead_mobility = mobility(sites, radius)                                 # [...] Direction-[beads] <- Force+[beads]
    # Over `Twist`, the space of all rates, each bead's commutator with the rate is that bead's velocity.
    rigid = sites.commutator(Twist)                                         # [...] Direction-[beads] <- Twist+
    # The forces that carry the beads along with the body at a given rate, summed into the forque on the
    # body: the adjugate carries forces back across the map that carried the rate out to velocities.
    resistance = rigid.adjugate()(bead_mobility.solve(rigid))               # [...] Forque- <- Twist+
    # Changing shape alone would push on the fluid too; the body moves at the rate whose forque cancels it.
    body_rate = -resistance.solve(rigid.adjugate()(bead_mobility.solve(shape_velocity)))   # [...] Twist+
    # The forces the beads push with under both motions together, balanced by construction.
    forces = bead_mobility.solve(rigid(body_rate) + shape_velocity)         # [...] Force+[beads]
    return body_rate, forces

## 2. Purcell's stroke beside a reciprocal stroke

Two swimmers stroke for three cycles. Purcell's swimmer swings its two hinges a quarter cycle apart, a loop in the space of hinge angles; the other swings both together, back and forth along a line. With nothing coasting, the body's rate is linear in the rate at which the shape changes, so how far a stroke carries the swimmer depends only on the path its shape takes, and going back along a path undoes it. The reciprocal swimmer lunges forward and back and ends every cycle where it started, while Purcell's swimmer steps steadily ahead.

The animation shows Purcell's swimmer with the forces its beads push with and the flow they drive, both swimmers' progress over the cycles, and the two strokes in the space of hinge angles.

In [ ]:
# The stroke's amplitude in radians, the cycles, the steps per cycle; the points the flow is drawn at; the steps
# per frame, and the duration of a frame.
amplitude = 1.05
n_cycles = 3
n_steps_per_cycle = 60
grid_x = np.linspace(-2.8, 2.8, 25)
grid_y = np.linspace(-1.9, 1.9, 17)
n_steps_per_frame = 3
duration_ms = 60

In [ ]:
def flow_field(sample_points: Point, bead_points: Point, bead_forces: Force, radius: float) -> Direction:
    """Fluid velocity `[...] Direction` at `sample_points` induced by `bead_forces` `[...] Force[beads]` at
    `bead_points` `[...] Point[beads]`: the map from the beads' forces to the flow at each sample point,
    applied to the forces, sums the flows of all the beads."""
    return stokeslet(sample_points[..., None], bead_points.batch(), radius).field(1)(bead_forces)   # [...] Direction-


def three_link_shape(
    hinge_half_span: float,
    centre_offsets: np.ndarray,
    arm_offsets: np.ndarray,
    angles: np.ndarray,
    rates: np.ndarray,
) -> tuple[Point, Direction]:
    """Bead points `[...] Point[beads]` and body-frame deformation velocities `[...] Direction[beads]`
    for a three-link Purcell swimmer with hinge angles `angles` `[..., 2]` and angular rates `rates` `[..., 2]`."""
    # The beads lie along the swimmer's axis, through the origin along x, at their offsets from the origin.
    along = mv.direction([1.0, 0.0])                                            # [] Point-
    centre_beads = origin + along * centre_offsets                              # [centre] Point-
    # A hinge is the meet of two planes through it, the plane along the swimmer's axis and the plane across it, a
    # bivector generating turns about itself.
    left_hinge = (mv.x + hinge_half_span * mv.w) ^ mv.y                         # [] Twist+
    right_hinge = (mv.x - hinge_half_span * mv.w) ^ mv.y                        # [] Twist+
    left_turn = (left_hinge * (angles[..., 0, None] / 2.0)).exp()               # [..., 1] Motor+
    right_turn = (right_hinge * (-angles[..., 1, None] / 2.0)).exp()            # [..., 1] Motor+

    left_rest = origin - along * (hinge_half_span + arm_offsets[::-1])          # [arm] Point-
    right_rest = origin + along * (hinge_half_span + arm_offsets)               # [arm] Point-
    left_beads = left_turn >> left_rest                                         # [..., arm] Point-
    right_beads = right_turn >> right_rest                                      # [..., arm] Point-

    # Each arm turns about its hinge at the hinge's rate, so a bead's velocity is its commutator with that
    # rate; the left hinge's angle counts the other way round. The centre link does not move in the body's
    # frame.
    left_vel = left_beads.commutator(-left_hinge * rates[..., 0, None])         # [..., arm] Direction-
    right_vel = right_beads.commutator(right_hinge * rates[..., 1, None])       # [..., arm] Direction-
    centre_vel = (angles[..., :1] * 0.0) * centre_beads.commutator(right_hinge)   # [..., centre] Direction-

    shape = angles.shape[:-1]
    sites = concatenate([left_beads, centre_beads.broadcast_to(shape + centre_beads.shape), right_beads], axis=-1)
    velocities = concatenate([left_vel, centre_vel, right_vel], axis=-1)
    return sites.field(), velocities.field()


n_steps = n_cycles * n_steps_per_cycle
dt = 2 * np.pi / n_steps_per_cycle
phases = np.arange(n_steps) * dt                                                # [steps]

# Purcell's swimmer takes its hinges around a circle; the other swings them together along a line.
angles = np.stack([
    np.stack([amplitude * np.cos(phases), amplitude * np.sin(phases)], axis=-1),
    np.stack([amplitude * np.cos(phases), -0.5 * amplitude * np.cos(phases)], axis=-1),
], axis=1)                                                                      # [steps, swimmers, 2]
rates = np.stack([
    np.stack([-amplitude * np.sin(phases), amplitude * np.cos(phases)], axis=-1),
    np.stack([-amplitude * np.sin(phases), 0.5 * amplitude * np.sin(phases)], axis=-1),
], axis=1)                                                                      # [steps, swimmers, 2]

sites, shape_velocity = three_link_shape(hinge_half_span, centre_offsets, arm_offsets, angles, rates)
body_rate, forces = swim(sites, shape_velocity, bead_radius)                    # [steps, swimmers] Twist+, Force+[beads]

# Each step moves the body in its own frame, so it multiplies the pose on the right. A running product
# puts later factors on the left; taking it over the steps' reverses and reversing the result puts them
# on the right.
start_motor = (mv.xy * np.array([0.0, 0.0])).exp()                              # [swimmers] Motor+
step_motors = (body_rate * (-dt / 2.0)).exp()                                   # [steps, swimmers] Motor+
motors = concatenate([start_motor[None], start_motor * step_motors.reverse().cumprod(axis=0).reverse()], axis=0)
positions = motors >> origin                                                    # [steps + 1, swimmers] Point-
# The beads and their forces carried from the body's frame into the world; a force is moved as its
# complement, a direction, and complemented again.
world_sites = motors[:-1] >> sites                                              # [steps, swimmers] Point-[beads]
world_forces = (motors[:-1] >> forces.dual()).dual()                            # [steps, swimmers] Force+[beads]


def flows(grid: Point):
    """Per frame, the flow Purcell's swimmer drives at the points of the grid."""
    for step in range(0, n_steps, n_steps_per_frame):
        yield flow_field(grid, world_sites[step, 0, None, None], world_forces[step, 0, None, None], bead_radius)


gx, gy = np.meshgrid(grid_x, grid_y, indexing="xy")
grid = mv.point(np.stack([gx, gy], axis=-1))                                    # [grid_y, grid_x] Point-
save_figure(render.draw_summary(angles, positions, world_sites, world_forces, grid, next(flows(grid))), "stokes_swimmer");

In [ ]:
show_animation(render.animate_swimmers(positions, world_sites, grid, flows(grid), n_steps_per_frame), "stokes_swimmer", duration_ms)

In [ ]:
# checks
self_mobility = stokeslet(origin, origin, bead_radius)
np.testing.assert_allclose((self_mobility - (2.0 / bead_radius) * Force.dual()).kernel, 0.0, atol=1e-12)

rigid = sites.commutator(Twist)
np.testing.assert_allclose(rigid.adjugate()(forces).kernel, 0.0, atol=1e-12)

cycle_motors = motors[n_steps_per_cycle::n_steps_per_cycle]                     # [cycles, swimmers] Motor+
np.testing.assert_allclose((cycle_motors[:, 1] - start_motor[1]).kernel, 0.0, atol=1e-4)
cycle_centres = positions[n_steps_per_cycle::n_steps_per_cycle]                 # [cycles, swimmers] Point-
purcell_x = (mv.x & (cycle_centres[:, 0] - origin)).to_array()
step_advances = np.diff(np.concatenate([[0.0], purcell_x]))
assert np.all(np.abs(step_advances) > 0.04)
np.testing.assert_allclose(step_advances, step_advances[0], rtol=1e-2)